# Lekcija 24: Projektivna geometrija

Lekcija 8 izgradila je hijerarhiju 2D transformacija &mdash; euklidskih, sličnosti i afinih &mdash; te istaknula da sve tri čuvaju paralelne pravce. Ova lekcija obrađuje sljedeći, najopćenitiji korak: **projektivnu transformaciju (homografiju)**, koja modelira ono što kamera stvarno radi kada ravnu površinu promatra pod kutom i koja *ne* čuva paralelnost. To nije pogreška &mdash; upravo je to pojava točke nedogleda i alat na kojem se temelje korekcija perspektive i spajanje slika.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Homogene koordinate

Predstavimo 2D točku $(x, y)$ vektorom s 3 komponente $(x, y, 1)$. Bilo koja matrica $H$ dimenzija $3\times3$ tada može djelovati na nju običnim množenjem matrica; pretvorba natrag u 2D znači dijeljenje trećom koordinatom:

$$\begin{bmatrix}x'\\y'\\w'\end{bmatrix} = H\begin{bmatrix}x\\y\\1\end{bmatrix}, \qquad (x_{\text{2D}}', y_{\text{2D}}') = \left(\frac{x'}{w'}, \frac{y'}{w'}\right)$$

Dvije velike prednosti: prvo, matrica $3\times3$ može predstavljati i translaciju (što nije moguće samo matricom $2\times2$, zbog čega je lekcija 8 trebala zaseban vektor translacije $t$). Drugo, upravo dijeljenje s $w'$ omogućuje homografiji modeliranje stvarnih *perspektivnih* učinaka &mdash; ne samo afinih transformacija iz lekcije 8, nego i fotografija na kojima paralelni pravci konvergiraju prema točki nedogleda, upravo kao kod stvarne kamere koja ravnu površinu promatra pod kutom. Također primijetite da $H$ i $cH$ (bilo koji skalarni višekratnik različit od nule) predstavljaju *potpuno istu* transformaciju jer dijeljenje poništava faktor mjerila &mdash; homografija ima samo 8 neovisnih stupnjeva slobode, a ne 9.

## Paralelni pravci prestaju biti paralelni

Ažurirana hijerarhija, koja proširuje tablicu iz lekcije 8:

| Transformacija | Matrični oblik | Čuva |
|---|---|---|
| Afina | $\begin{bmatrix}a&b&t_x\\c&d&t_y\\0&0&1\end{bmatrix}$ | paralelnost |
| **Projektivna** | $\begin{bmatrix}a&b&t_x\\c&d&t_y\\g&h&1\end{bmatrix}$ | samo pravce |

Jedina je razlika donji redak $(g, h)$ različit od nule &mdash; a već to narušava paralelnost. Pokazujemo izravno: uzimamo dva paralelna vodoravna pravca i primjenjujemo homografiju s donjim retkom različitim od nule.

In [ ]:
H = np.array([
    [1,     0.2,   0],
    [0.1,   1,     0],
    [0.02,  0.015, 1],
], dtype=np.float64)

def apply_homography(points, H):
    homogeneous = np.hstack([points, np.ones((len(points), 1))])
    transformed = (H @ homogeneous.T).T
    return transformed[:, :2] / transformed[:, 2:3]

line1 = np.array([[0, 0], [1, 0]], dtype=np.float64)   # y = 0
line2 = np.array([[0, 1], [1, 1]], dtype=np.float64)   # y = 1, parallel to line1

l1_transformed = apply_homography(line1, H)
l2_transformed = apply_homography(line2, H)

print('line 1, transformed:', l1_transformed)
print('line 2, transformed:', l2_transformed)

def extended(p1, p2, length):
    direction = (p2 - p1)
    return p1 - length * direction, p2 + length * direction

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

for line, color in [(line1, 'tab:blue'), (line2, 'tab:orange')]:
    a, b = extended(line[0], line[1], length=3)
    axes[0].plot([a[0], b[0]], [a[1], b[1]], color=color)
    axes[0].plot(line[:, 0], line[:, 1], 'o', color=color)
axes[0].set_xlim(-2, 4)
axes[0].set_ylim(-2, 3)
axes[0].set_title('Before: two parallel lines')

for line, color in [(l1_transformed, 'tab:blue'), (l2_transformed, 'tab:orange')]:
    a, b = extended(line[0], line[1], length=60)
    axes[1].plot([a[0], b[0]], [a[1], b[1]], color=color)
    axes[1].plot(line[:, 0], line[:, 1], 'o', color=color)
axes[1].set_xlim(-5, 55)
axes[1].set_ylim(-2, 8)
axes[1].set_title('After: a homography with a nonzero bottom row')

plt.tight_layout()
plt.show()

## Točke nedogleda, izračunate na dva načina

Gdje se ta dva pravca zapravo sijeku? To možemo pronaći na dva načina: izravnim presijecanjem transformiranih segmenata pravaca ili &mdash; elegantnije &mdash; transformiranjem *točke u beskonačnosti* u zajedničkom smjeru izvornih pravaca $(1, 0)$, predstavljene u homogenim koordinatama kao $(1, 0, 0)$ (treća koordinata različita od nule dala bi konačnu točku; nula znači „beskonačno daleko”). Primjena $H$ na tu točku u beskonačnosti trebala bi završiti upravo na točki nedogleda.

In [ ]:
def line_intersection(p1, p2, p3, p4):
    A = np.array([[p2[0] - p1[0], -(p4[0] - p3[0])],
                  [p2[1] - p1[1], -(p4[1] - p3[1])]])
    b = np.array([p3[0] - p1[0], p3[1] - p1[1]])
    t = np.linalg.solve(A, b)[0]
    return p1 + t * (p2 - p1)

vanishing_point_direct = line_intersection(l1_transformed[0], l1_transformed[1],
                                            l2_transformed[0], l2_transformed[1])

point_at_infinity = np.array([1.0, 0.0, 0.0])  # direction (1,0), infinitely far away
transformed_infinity = H @ point_at_infinity
vanishing_point_via_infinity = transformed_infinity[:2] / transformed_infinity[2]

print('vanishing point (line intersection): ', vanishing_point_direct)
print('vanishing point (H @ infinity trick):', vanishing_point_via_infinity)

## Pravci u homogenim koordinatama

Točke nisu jedino što homogene koordinate elegantno predstavljaju &mdash; pravac $ax+by+c=0$ jednako prirodno predstavlja vektor s 3 komponente $l=(a,b,c)$, pri čemu točka $p=(x,y,1)$ leži na njemu upravo kada je skalarni umnožak $l\cdot p= l^\top p = 0$. Iz vektorskog umnoška slijede dvije korisne posljedice: pravac kroz dvije točke je $l = p_1\times p_2$ (okomit je na obje, pa su oba skalarna umnoška nula), a prema istoj logici u obrnutom smjeru, sjecište dvaju pravaca je $p = l_1\times l_2$. Gornji `line_intersection` ručno je riješio linearni sustav $2\times2$; vektorski umnožak radi isto u jednom retku koda.

In [ ]:
def line_through(p1, p2):
    return np.cross([p1[0], p1[1], 1.0], [p2[0], p2[1], 1.0])

def intersect(l1, l2):
    p = np.cross(l1, l2)
    return p[:2] / p[2]

line1_h = line_through(l1_transformed[0], l1_transformed[1])
line2_h = line_through(l2_transformed[0], l2_transformed[1])
vanishing_point_cross = intersect(line1_h, line2_h)

print('vanishing point (cross-product trick):', vanishing_point_cross)

Rezultat se potpuno podudara s dvjema prethodnim metodama. Pokazuje se da točke i pravci (u 2D ravnini) imaju potpuno simetrične uloge u homogenim koordinatama &mdash; ista operacija vektorskog umnoška gradi pravac iz dviju točaka ili točku iz dvaju pravaca.

## Kada homografija doista vrijedi?

Homografija točno povezuje dvije fotografije u upravo dvije situacije: fotografiranje ravne površine iz dvaju različitih položaja (slučaj korišten u nastavku) ili fotografiranje *bilo kojeg* 3D prizora, ravnog ili ne, iz istog položaja kamere uz samo njezinu rotaciju između snimki. Drugi slučaj omogućuje spajanje panorama iz lekcije 25 na običnim, neravnim prizorima &mdash; za kameru koja se translacijski pomiče ili stvarno 3D prizor promatran iz dvaju različitih položaja ne postoji jedna homografija koja točno povezuje dvije slike.

## Korekcija perspektive: rektifikacija fotografiranog dokumenta

Najčešća praktična uporaba homografije: uz 4 korespondencije točaka (npr. 4 kuta dokumenta, odabrana klikom korisnika ili automatski pronađena), `cv2.getPerspectiveTransform` određuje jedinstvenu homografiju koja jedan skup od 4 točke preslikava u drugi, a `cv2.warpPerspective` je primjenjuje. Trapezni oblik fotografiranog pravokutnika &mdash; rubovi koji su u stvarnosti bili paralelni vidljivo konvergiraju na fotografiji &mdash; naziva se **trapezna perspektivna deformacija (keystoning)**. To je upravo prethodni učinak točke nedogleda, sada primijenjen na dokument umjesto na dva apstraktna pravca. (U nastavku se `H_rectify` računa iz korespondencija točaka u obrnutom redoslijedu, ali jednak je `np.linalg.inv(H_distort)`, jer su poništavanje homografije i invertiranje njezine matrice ista operacija.)

In [ ]:
document = np.full((300, 220, 3), 255, dtype=np.uint8)
cv2.rectangle(document, (20, 20), (200, 280), (0, 0, 0), 3)
cv2.putText(document, 'HELLO', (30, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 3)

h, w = document.shape[:2]
corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
photographed_corners = np.float32([[45, 15], [175, 25], [210, 290], [5, 280]])  # strong keystone: top edge narrower than bottom

H_distort = cv2.getPerspectiveTransform(corners, photographed_corners)
photographed = cv2.warpPerspective(document, H_distort, (w, h))

H_rectify = cv2.getPerspectiveTransform(photographed_corners, corners)
rectified = cv2.warpPerspective(photographed, H_rectify, (w, h))

fig, axes = plt.subplots(1, 3, figsize=(9, 4))
for ax, im, title in zip(axes, [document, photographed, rectified],
                          ['Original document', 'Photographed at an angle\n(simulated)', 'Rectified\n(from 4 corner clicks)']):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

error = np.abs(rectified.astype(int) - document.astype(int))
print(f'mean abs pixel difference after distort-then-rectify round trip: {error.mean():.2f}  (small resampling loss only)')

## Određivanje homografije iz mnogo korespondencija

4 korespondencije točno određuju 8 stupnjeva slobode homografije, bez prostora za pogrešku. Uz *više* od 4 (obično iz automatskog podudaranja značajki, lekcija 20), problem postaje preodređena prilagodba najmanjim kvadratima &mdash; algoritam izravne linearne transformacije (DLT) koji implementira `cv2.findHomography`, uz opcionalnu uporabu RANSAC-a (lekcija 23) za odbacivanje pogrešnih korespondencija (pogrešno podudarenih značajki) kao izdvojenih vrijednosti. `cv2.findHomography` prije rješavanja također interno normalizira koordinate točaka radi numeričke uvjetovanosti &mdash; isti postupak izričito prikazan u algoritmu s 8 točaka iz lekcije 28.

In [ ]:
rng = np.random.default_rng(0)
H_true = np.array([[1, 0.2, 10], [0.05, 1, 5], [0.0008, 0.0003, 1]])

pts1 = rng.uniform(0, 200, (30, 2))
pts2 = apply_homography(pts1, H_true)

H_estimated, inlier_mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, 3.0)

print('true homography (scale-normalized):\n', np.round(H_true / H_true[2, 2], 4))
print('estimated homography:\n', np.round(H_estimated, 4))
print(f'inliers: {int(inlier_mask.sum())} / {len(inlier_mask)}')

Uz ispravne korespondencije `findHomography` određuje `H_true` gotovo potpuno točno &mdash; upravo je to posljednji korak tipičnog postupka spajanja slika: detektirati i podudariti SIFT značajke između dviju preklapajućih fotografija (lekcija 20), a zatim odrediti homografiju koja jednu poravnava s drugom.

## RANSAC u praksi: procjena homografije uz pogrešna podudaranja

Iako gornji kod predaje `cv2.RANSAC` funkciji `findHomography`, uz ispravne korespondencije nema ničega za odbacivanje &mdash; ne stvara vidljivu razliku. Kakav učinak RANSAC zapravo ima? Budući da stvarno podudaranje značajki (lekcija 20) neizbježno daje i pogrešna podudaranja, čak i nakon testa omjera, trebamo način odbacivanja takvih izdvojenih vrijednosti. Radi ilustracije dodajemo 15 namjerno potpuno pogrešnih korespondencija uz 30 stvarnih i uspoređujemo RANSAC prilagodbu s običnom prilagodbom najmanjim kvadratima na potpuno istim onečišćenim podacima.

In [ ]:
pts1_bad = rng.uniform(0, 200, (15, 2))
pts2_bad = rng.uniform(0, 300, (15, 2))  # unrelated -- simulated bad matches

pts1_contaminated = np.vstack([pts1, pts1_bad])
pts2_contaminated = np.vstack([pts2, pts2_bad])

H_ransac, mask = cv2.findHomography(pts1_contaminated, pts2_contaminated, cv2.RANSAC, 3.0)
H_plain, _ = cv2.findHomography(pts1_contaminated, pts2_contaminated, 0)  # method=0: plain least squares, no outlier rejection

def mean_reprojection_error(H, pts1, pts2):
    return np.linalg.norm(apply_homography(pts1, H) - pts2, axis=1).mean()

print(f'inliers found by RANSAC: {int(mask.sum())} / {len(mask)}  (30 correspondences were genuinely correct)')
print()
print(f'mean reprojection error on the TRUE inliers:')
print(f'  RANSAC fit: {mean_reprojection_error(H_ransac, pts1, pts2):.2e} pixels')
print(f'  plain fit:  {mean_reprojection_error(H_plain, pts1, pts2):.2e} pixels')

RANSAC pronalazi točno 30 stvarnih korespondencija i rekonstruira homografiju gotovo do strojne preciznosti. Obična prilagodba najmanjim kvadratima na potpuno istim podacima daje pogrešku veću za mnogo redova veličine &mdash; praktički beskorisnu za sve što zahtijeva pikselnu preciznost &mdash; jer je 15 pogrešnih podudaranja od 45 (33 %) bilo više nego dovoljno za primjetno narušavanje neponderirane prilagodbe zbrojem kvadrata.

## Pogled unaprijed: projektivna geometrija u 3D prostoru

Sve u ovoj lekciji bilo je 2D-u-2D: homografija povezuje dvije ravnine (slike ili ravne površine u svijetu). Lekcija 26 proširuje isti projektivni postupak za još jednu dimenziju, izvodeći projekcijsku matricu kamere koja uopće preslikava 3D prizor na 2D sliku.

### Zadaci

1. Dodajte Gaussov šum srednje vrijednosti nula (npr. standardne devijacije 2 piksela) u `pts2` prije pozivanja `cv2.findHomography`. Koliko procijenjena homografija odstupa od `H_true` i smanjuje li povećanje broja korespondencija (primjerice s 30 na 200) to odstupanje?
2. Povećavajte broj dodanih pogrešnih podudaranja (`pts1_bad`/`pts2_bad`) od 15 nadalje dok RANSAC ne počne zakazivati. Koristeći formulu broja iteracija iz lekcije 23 (minimalna veličina uzorka 4 za homografiju), koliki je približan udio izdvojenih vrijednosti i koliko bi iteracija teorijski trebalo za zadržavanje 99 % pouzdanosti uspjeha?
3. U demonstraciji točke nedogleda promijenite dva pravca tako da budu *okomiti* umjesto vodoravni (npr. `x=0` i `x=1`), a zatim predvidite i provjerite gdje završava njihova točka nedogleda pomoću postupka s točkom u beskonačnosti smjera `(0, 1, 0)`.